# D3 companion — Run a real language model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/5x5x5x5/taihls/blob/course-curriculum/notebooks/d3-run-an-llm.ipynb)

In the chapter we built a tiny bigram model by hand. Here we run a **real** (small) large language model, `distilgpt2`, with Hugging Face `transformers` on a GPU: we generate text and inspect the next-token **probability distribution** it predicts — the same idea as our toy model, just learned by a neural network on a flood of text.

> Runs best on Colab with a GPU runtime (*Runtime → Change runtime type → GPU*).

## 1. Install dependencies

In [ ]:
!pip install -q transformers torch

## 2. Check for a GPU

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)
if device == 'cpu':
    print('No GPU found — it will still run, just slower. '
          'On Colab: Runtime → Change runtime type → GPU.')

## 3. Load the model

`distilgpt2` is a distilled (smaller, faster) version of GPT-2. It is a true next-token predictor, just like our bigram model — only it looks back over many tokens and learned its probabilities from a neural network.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = 'distilgpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()
print(f'Loaded {model_name} with {model.num_parameters():,} parameters.')

## 4. Generate text

Give the model a prompt and let it continue — it repeatedly predicts and samples the next token, exactly like our `generate` function in the chapter.

In [ ]:
prompt = 'The patient was prescribed'
inputs = tokenizer(prompt, return_tensors='pt').to(device)

output = model.generate(
    **inputs,
    max_new_tokens=30,
    do_sample=True,
    top_k=50,
    temperature=0.8,
    pad_token_id=tokenizer.eos_token_id,
)
print(tokenizer.decode(output[0], skip_special_tokens=True))

## 5. Inspect the next-token probability distribution

This is the heart of the chapter made real: for one prompt, what probability does the model assign to each possible next token? We show the top 10.

In [ ]:
import torch.nn.functional as F

prompt = 'The doctor checked the'
inputs = tokenizer(prompt, return_tensors='pt').to(device)

with torch.no_grad():
    logits = model(**inputs).logits          # scores for every token
next_token_logits = logits[0, -1, :]         # scores for the *next* token
probs = F.softmax(next_token_logits, dim=-1) # turn scores into a distribution

top = torch.topk(probs, 10)
print(f'Most likely next tokens after: {prompt!r}\n')
for prob, idx in zip(top.values, top.indices):
    token = tokenizer.decode(idx).strip() or '(space)'
    print(f'  {token:>12}  p = {prob.item():.3f}')

Notice the probabilities form a **distribution over the whole vocabulary** (they sum to 1 across all ~50,000 tokens). Generating text just means sampling from this distribution, appending the token, and repeating — the exact loop from the chapter, scaled up.

## 6. Your turn

- Change `prompt` in section 4 and see how the continuation changes.
- Lower the `temperature` (e.g. 0.3) and raise it (e.g. 1.2). What happens to how 'safe' vs. 'wild' the text gets?
- Pick a prompt that asks for a **fact** (e.g. `'The capital of Australia is'`) and inspect the top tokens in section 5. Is the model confident? Is it right? (This previews the **hallucination** chapter, D4.)
- Try a larger model such as `'gpt2'` in section 3 and compare.